# DeepSeek-V2-Lite Trace Recording

**Runtime:** A100 GPU (Colab Pro/Pro+)

Records expert activation traces from DeepSeek-V2-Lite (16B params,
64 routed experts + shared experts, top-6 routing per layer).

**Important:** Select Runtime → Change runtime type → A100 GPU before running.

In [ ]:
# Install dependencies
# DeepSeek-V2-Lite's custom code requires transformers<4.46 (is_torch_fx_available removed in 4.46)
!pip install -q torch "transformers>=4.40,<4.46" accelerate lark>=1.1 huggingface_hub

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

import transformers
print(f'transformers version: {transformers.__version__}')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/moe-sched-paper'
os.makedirs(f'{DRIVE_ROOT}/traces', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

In [ ]:
# Unzip moe_sched from Google Drive
!unzip -qo /content/drive/MyDrive/moe-sched-paper/moe_sched.zip -d /content/

import sys
sys.path.insert(0, '/content')

from moe_sched import MoESched
print('MoE-Sched imported successfully')

## 1. Load DeepSeek-V2-Lite

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

MODEL_NAME = 'deepseek-ai/DeepSeek-V2-Lite'

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loading model: {MODEL_NAME}')
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
print(f'Loaded in {time.time() - t0:.1f}s')
print(f'Device map keys: {list(model.hf_device_map.keys())[:10]}...')

## 2. Inspect Model Structure

In [ ]:
# Discover the MoE layer structure
# DeepSeek-V2-Lite may use different attribute names than Mixtral

base = getattr(model, 'model', model)
layers = getattr(base, 'layers', None)
if layers is None:
    # Try transformer.h (GPT family)
    layers = getattr(getattr(model, 'transformer', None), 'h', None)

print(f'Number of layers: {len(layers)}')
print(f'Layer type: {type(layers[0]).__name__}')
print(f'Layer children: {[n for n, _ in layers[0].named_children()]}')

In [ ]:
# Find the MoE block — DeepSeek-V2-Lite has mixed dense + MoE layers
# Not all layers have MoE, so we scan to find the first one

moe_layer_indices = []
moe_attr = None
gate_attr = None

for idx, layer in enumerate(layers):
    mlp = getattr(layer, 'mlp', None)
    if mlp is None:
        continue
    # Check if this MLP is an MoE block (has gate/experts)
    has_gate = any(hasattr(mlp, a) for a in ['gate', 'router', 'gating_network'])
    has_experts = hasattr(mlp, 'experts')
    if has_gate or has_experts:
        moe_layer_indices.append(idx)
        if moe_attr is None:
            moe_attr = 'mlp'
            # Find the gate attribute name
            for gname in ['gate', 'router', 'gating_network', 'gate_proj']:
                if hasattr(mlp, gname):
                    gate_attr = gname
                    break

print(f'MoE layers: {len(moe_layer_indices)} out of {len(layers)} total')
print(f'MoE layer indices: {moe_layer_indices[:10]}{"..." if len(moe_layer_indices) > 10 else ""}')

if moe_layer_indices:
    sample_moe = getattr(layers[moe_layer_indices[0]], moe_attr)
    print(f'MoE block type: {type(sample_moe).__name__}')
    print(f'MoE children: {[(n, type(c).__name__) for n, c in sample_moe.named_children()]}')
    print(f'Gate attribute: {gate_attr}')
    if gate_attr:
        gate_module = getattr(sample_moe, gate_attr)
        print(f'Gate type: {type(gate_module).__name__}')
        if hasattr(gate_module, 'weight'):
            print(f'Gate weight shape: {gate_module.weight.shape}')
            num_experts = gate_module.weight.shape[0]
            print(f'Number of experts: {num_experts}')
else:
    print('WARNING: No MoE layers found!')
    print(f'Layer 0 mlp type: {type(getattr(layers[0], "mlp", None)).__name__}')
    print(f'Layer 0 mlp attrs: {[a for a in dir(getattr(layers[0], "mlp", None)) if not a.startswith("_")]}')

In [ ]:
# Get top_k and num_layers from the MoE config
sample_moe = getattr(layers[moe_layer_indices[0]], moe_attr)
top_k = getattr(sample_moe, 'top_k', None) or getattr(sample_moe, 'num_experts_per_tok', None)
if top_k is None:
    # Try model config
    top_k = getattr(model.config, 'num_experts_per_tok', 6)
print(f'Top-k: {top_k}')

num_layers = len(layers)
num_moe_layers = len(moe_layer_indices)
print(f'Model: {num_layers} layers total, {num_moe_layers} MoE layers')
print(f'Dense layers: {[i for i in range(num_layers) if i not in moe_layer_indices][:10]}')

In [ ]:
# Probe the gate output format with a test input on the first MoE layer
debug_output = [None]

def debug_hook(module, input, output):
    if debug_output[0] is None:
        debug_output[0] = output
        print(f'Gate output type: {type(output)}')
        if isinstance(output, tuple):
            print(f'Tuple length: {len(output)}')
            for i, o in enumerate(output):
                if hasattr(o, 'shape'):
                    print(f'  [{i}] {type(o).__name__} shape={o.shape}')
                else:
                    print(f'  [{i}] {type(o).__name__}')
        elif hasattr(output, 'shape'):
            print(f'Tensor shape: {output.shape}')

# Hook the gate of the first MoE layer
first_moe_layer = layers[moe_layer_indices[0]]
gate = getattr(getattr(first_moe_layer, moe_attr), gate_attr)
print(f'Hooking gate on layer {moe_layer_indices[0]}: {type(gate).__name__}')

h = gate.register_forward_hook(debug_hook)
inputs = tokenizer('Hello', return_tensors='pt').to(model.device)
with torch.no_grad():
    _ = model(**inputs)  # forward pass only, no generation args
h.remove()
print('\nGate probe complete — use the output format above to write the trace hook')

## 3. Install Trace Hooks

**After running the probe cell above**, adapt the hook below based on the gate output format.

Common formats:
- Mixtral: `(router_logits, topk_weights, topk_ids)` — use `output[2]` for expert IDs
- DeepSeek: may vary — check the probe output

In [ ]:
import json

trace_data = []
token_counter = [0]
_debug_printed = [False]

def make_gate_hook(layer_idx):
    """Hook the gate module to capture router selections."""
    def hook_fn(module, input, output):
        # Debug: print gate output format on first call
        if not _debug_printed[0]:
            _debug_printed[0] = True
            print(f'[DEBUG] Gate output type: {type(output).__name__}')
            if isinstance(output, tuple):
                print(f'[DEBUG] Tuple len={len(output)}')
                for i, o in enumerate(output):
                    if hasattr(o, 'shape'):
                        print(f'  [{i}] shape={o.shape} dtype={o.dtype}')
                    else:
                        print(f'  [{i}] {type(o).__name__}: {o}')
            elif hasattr(output, 'shape'):
                print(f'[DEBUG] Tensor shape={output.shape} dtype={output.dtype}')

        topk_ids = None
        topk_weights = None
        tk = top_k or 6

        if isinstance(output, tuple):
            # Check each element for expert indices (integer tensor)
            for i, o in enumerate(output):
                if hasattr(o, 'dtype') and o.dtype in (torch.int64, torch.int32, torch.long):
                    topk_ids = o
                elif hasattr(o, 'dtype') and o.is_floating_point() and topk_weights is None:
                    if o.shape[-1] == tk or (topk_ids is not None and o.shape == topk_ids.shape):
                        topk_weights = o
            # Fallback: if no integer tensor, first element is likely logits
            if topk_ids is None and len(output) > 0:
                logits = output[0] if hasattr(output[0], 'shape') else None
                if logits is not None:
                    topk_weights, topk_ids = torch.topk(
                        torch.softmax(logits.float(), dim=-1), tk, dim=-1
                    )
        elif hasattr(output, 'shape'):
            # Single tensor — router logits
            topk_weights, topk_ids = torch.topk(
                torch.softmax(output.float(), dim=-1), tk, dim=-1
            )

        if topk_ids is None:
            return  # skip this call if we can't parse

        # Flatten to 2D: [tokens, top_k]
        if topk_ids.dim() == 1:
            topk_ids = topk_ids.unsqueeze(0)
            if topk_weights is not None:
                topk_weights = topk_weights.unsqueeze(0)

        for i in range(topk_ids.shape[0]):
            entry = {
                't': token_counter[0] + i,
                'l': layer_idx,
                'e': topk_ids[i].cpu().tolist(),
            }
            if topk_weights is not None:
                entry['s'] = [round(s, 4) for s in topk_weights[i].cpu().tolist()]
            trace_data.append(entry)
    return hook_fn

# Install hooks ONLY on MoE layers (skip dense layers)
hooks = []
_debug_printed[0] = False
for idx in moe_layer_indices:
    layer = layers[idx]
    moe = getattr(layer, moe_attr)
    g = getattr(moe, gate_attr, None)
    if g is not None:
        h = g.register_forward_hook(make_gate_hook(idx))
        hooks.append(h)
    else:
        print(f'WARNING: No gate found on MoE layer {idx}')

print(f'Installed hooks on {len(hooks)} / {len(moe_layer_indices)} MoE layers')
print('First call will print [DEBUG] gate output format')

## 4. Run Inference and Capture Traces

In [ ]:
trace_data.clear()
token_counter[0] = 0

prompts = [
    'Explain the difference between transformers and RNNs.',
    'Write a Python function to sort a list using merge sort.',
    'What are the main challenges in training large language models?',
    'Describe the architecture of a Mixture-of-Experts model.',
    'How does expert caching improve MoE inference performance?',
]

print(f'Running inference on {len(prompts)} prompts...')
for i, prompt in enumerate(prompts):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    token_counter[0] = 0
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
        )
    num_tokens = outputs.shape[1]
    print(f'  Prompt {i+1}: {num_tokens} tokens, {len(trace_data)} trace entries so far')

print(f'\nTotal trace entries: {len(trace_data)}')

# Remove hooks
for h in hooks:
    h.remove()
print('Hooks removed')

## 5. Analyze and Save

In [ ]:
from collections import Counter

expert_counts = Counter()
for entry in trace_data:
    for e in entry['e']:
        expert_counts[e] += 1

print('Expert activation distribution (top 15):')
for expert_id, count in expert_counts.most_common(15):
    pct = 100 * count / sum(expert_counts.values())
    print(f'  Expert {expert_id}: {count} activations ({pct:.1f}%)')

print(f'\nTotal unique experts activated: {len(expert_counts)}')
print(f'Total activations: {sum(expert_counts.values())}')

In [ ]:
# Save trace to Google Drive
trace_path = f'{DRIVE_ROOT}/traces/deepseek_v2_lite_sample.jsonl'

# Determine num_experts from data
all_experts = set()
for entry in trace_data:
    all_experts.update(entry['e'])
detected_num_experts = max(all_experts) + 1 if all_experts else 0
detected_top_k = len(trace_data[0]['e']) if trace_data else 0

with open(trace_path, 'w') as f:
    f.write(json.dumps({
        'model_name': MODEL_NAME,
        'num_layers': num_layers,
        'num_experts': detected_num_experts,
        'top_k': detected_top_k,
        'num_entries': len(trace_data),
    }) + '\n')
    for entry in trace_data:
        f.write(json.dumps(entry) + '\n')

print(f'Trace saved to: {trace_path}')
print(f'File size: {os.path.getsize(trace_path) / 1e6:.1f} MB')

## 6. Test MoE-Sched Policies on DeepSeek Trace

In [ ]:
from moe_sched import MoESched
from moe_sched.compiler import compile_policy
from moe_sched.runtime.hooks import build_hook

# Test a range of capacities
print(f'Capacity sweep (LRU, DeepSeek-V2-Lite trace)')
print(f'{"Cap":>5} {"Hits":>8} {"Misses":>8} {"Hit Rate":>10} {"Evictions":>10}')
print('-' * 45)

for cap in [4, 8, 16, 32, 64]:
    sched = MoESched()
    ir = (
        sched.build(f'lru_{cap}')
        .cache(capacity=cap, eviction='lru')
        .prefetch(strategy='none', budget=min(4, cap))
        .schedule(mode='gpu_only')
        .done()
    )
    hook = build_hook(compile_policy(ir))
    for entry in trace_data:
        hook.on_layer(layer_idx=entry['l'], selected_experts=entry['e'], scores=entry.get('s'))
    s = hook.stats_snapshot()
    h, m = s['cache']['hits'], s['cache']['misses']
    hr = h / max(1, h + m)
    ev = s['cache']['evictions']
    print(f'{cap:>5} {h:>8} {m:>8} {hr:>9.1%} {ev:>10}')

print('\nDone! Download traces from Google Drive for local analysis.')

## Next Steps

- [ ] Download `deepseek_v2_lite_sample.jsonl` from Drive
- [ ] Run full evaluation with `scripts/run_eval.py`
- [ ] Compare Mixtral vs DeepSeek expert activation patterns
- [ ] Run with larger workloads (ShareGPT, LMSYS)